# Building a multi-task model to predict age and gender from photo - transfer learning from `resnet18`

- **Dataset**: UTKFace
UTKFace dataset is a large-scale face dataset with long age span (range from 0 to 116 years old). The dataset consists of over 20,000 face images with annotations of age, gender, and ethnicity. The images cover large variation in pose, facial expression, illumination, occlusion, resolution, etc. This dataset could be used on a variety of tasks, e.g., face detection, age estimation, age progression/regression, landmark localization, etc.


In [8]:
!pip install torchmetrics

In [9]:
import os
import copy
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from torchmetrics.classification import Accuracy
from torchmetrics.regression import MeanAbsoluteError
from sklearn.model_selection import train_test_split
from pathlib import Path
from PIL import Image
from google.colab import drive

In [10]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **1. Data**

In [ ]:
class UTKFaceDataset(Dataset):
    def __init__(self, directories, transform=None):
        self.image_paths = []
        self.ages = []
        self.genders = []
        self.transform = transform

        for directory in directories:
            for root, _, files in os.walk(directory):
                for file in files:
                    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        parts = file.split('_')
                        if len(parts) >= 3:
                            try:
                                age = float(parts[0])
                                gender = float(parts[1])
                                self.image_paths.append(os.path.join(root, file))
                                self.ages.append(age)
                                self.genders.append(gender)
                            except ValueError:
                                continue
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        age = torch.tensor(self.ages[index], dtype=torch.float32)
        gender = torch.tensor(self.genders[index], dtype=torch.long)

        return image, age, gender

In [ ]:
data_dirs = ['/content/drive/MyDrive/data/faces/crop_part1', '/content/drive/MyDrive/data/faces/UTKFace']

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = UTKFaceDataset(directories=data_dirs, transform=base_transform)
indices = list(range(len(full_dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=full_dataset.genders, random_state=42
)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=32, shuffle=True, num_workers=os.cpu_count())
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=32, shuffle=False, num_workers=os.cpu_count())

device = "cuda" if torch.cuda.is_available() else "cpu"

# **2. Model**

In [ ]:
class MultiTaskResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        for param in self.backbone.parameters():
            param.requires_grad = False

        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.gender_head = nn.Linear(num_features, 1)
        self.age_head = nn.Linear(num_features, 1)

    def forward(self, x):
        features = self.backbone(x)
        gender_out = self.gender_head(features).squeeze()
        age_out = self.age_head(features).squeeze()

        return gender_out, age_out

In [ ]:
model = MultiTaskResNet().to(device)

gender_accuracy = Accuracy(task='binary').to(device)
age_mae = MeanAbsoluteError().to(device)

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, num_epochs=5, alpha=0.01, best_loss=float('inf')):
    criterion_gender = nn.BCEWithLogitsLoss()
    criterion_age = nn.MSELoss()

    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch+1}/{num_epochs}")

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                loader = train_loader
            else:
                model.eval()
                loader = val_loader

            running_total_loss = 0.0

            gender_accuracy.reset()
            age_mae.reset()

            for images, ages, genders in loader:
                images = images.to(device)
                ages = ages.to(device)
                genders = genders.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    gender_preds, age_preds = model(images)

                    loss_gender = criterion_gender(gender_preds, genders.float())
                    loss_age = criterion_age(age_preds, ages)
                    loss_total = loss_gender + (alpha * loss_age)

                    if phase == 'train':
                        loss_total.backward()
                        optimizer.step()
                running_total_loss += loss_total.item() * images.size(0)

                gender_accuracy.update(gender_preds, genders)
                age_mae.update(age_preds, ages)

            dataset_size = len(loader.dataset)
            epoch_loss = running_total_loss / dataset_size

            epoch_gender_acc = gender_accuracy.compute().item()
            epoch_age_mae = age_mae.compute().item()

            print(f"{phase.upper()} | Total Loss: {epoch_loss:.4f} | Gender Acc: {epoch_gender_acc:.2%} | Age MAE: {epoch_age_mae:.1f} years")

            if phase == 'val' and epoch_loss < best_loss:
                best_loss = epoch_loss
                best_weights = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_weights)
    return model, best_loss

In [ ]:
optimizer_extract = torch.optim.Adam(
    params=list(model.gender_head.parameters()) + list(model.age_head.parameters()),
    lr=1e-3
)

trained_model, best_val_loss = train_model(model=model, train_loader=train_loader, val_loader=val_loader, optimizer=optimizer_extract, num_epochs=5, alpha=0.01, best_loss=float('inf'))

for name, child in model.backbone.named_children():
    if name in ['layer3', 'layer4']:
        for param in child.parameters():
            param.requires_grad = True

optimizer_fine = torch.optim.Adam([
    {'params': model.backbone.layer3.parameters(), 'lr': 1e-5},
    {'params': model.backbone.layer4.parameters(), 'lr': 1e-5},
    {'params': model.gender_head.parameters(), 'lr': 1e-4},
    {'params': model.age_head.parameters(), 'lr': 1e-4}
])

model, best_val_loss = train_model(model=model, train_loader=train_loader, val_loader=val_loader, optimizer=optimizer_fine, num_epochs=10, alpha=0.01, best_loss=best_val_loss)



In [ ]:
models_folder = Path("models")
models_folder.mkdir(parents=True, exist_ok=True)
models_name = "UTKFacesMultiTaskModel.pth"

torch.save(obj=model.state_dict(), f=models_folder/models_name)